# 03. Baseline Models Training

This notebook trains baseline models:
- **Model A**: Text-only BERT classifier
- **Model B**: Image-only ViT classifier
- **Model C**: Standard multimodal fusion (without consistency)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').absolute()))

import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

from src.utils import set_seed, get_device, ensure_dir, load_config
from src.dataset import MultimodalFakeNewsDataset, stratified_split
from src.preprocessing import MultimodalPreprocessor
from src.classifier import MultimodalClassifier, ClassifierConfig
from src.train import Trainer, TrainingConfig
from torch.utils.data import DataLoader

## Configuration

In [ ]:
# Set random seed
set_seed(42)

# Device
device = get_device()
print(f"Using device: {device}")

# Paths - UPDATE THESE FOR YOUR DATASET
DATA_DIR = Path('data/processed')
IMAGES_DIR = Path('data/fakeddit/images')
CHECKPOINT_DIR = Path('artifacts/checkpoints')
ensure_dir(CHECKPOINT_DIR)

# Load config
config = load_config('configs/experiment.yaml')
print(f"Loaded config from configs/experiment.yaml")

## Load Dataset

In [ ]:
# Load train/val/test CSVs
train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df = pd.read_csv(DATA_DIR / 'val.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

In [ ]:
# Create dataset samples
from src.dataset import Sample

def create_samples(df, images_dir):
    samples = []
    for _, row in df.iterrows():
        text = str(row.get('clean_title', row.iloc[0]))
        image_id = str(row.get('image_id', ''))
        label = int(row.get('binary_label', row.iloc[-1]))
        
        image_path = images_dir / f"{image_id}.jpg" if image_id else None
        
        sample = Sample(
            sample_id=str(row.name),
            text=text,
            image_path=str(image_path) if image_path and image_path.exists() else None,
            label=label,
            metadata={}
        )
        samples.append(sample)
    return samples

train_samples = create_samples(train_df, IMAGES_DIR)
val_samples = create_samples(val_df, IMAGES_DIR)
test_samples = create_samples(test_df, IMAGES_DIR)

train_dataset = MultimodalFakeNewsDataset(train_samples)
val_dataset = MultimodalFakeNewsDataset(val_samples)
test_dataset = MultimodalFakeNewsDataset(test_samples)

print(f"Datasets created:")
print(f"  Train: {len(train_dataset)}")
print(f"  Val: {len(val_dataset)}")
print(f"  Test: {len(test_dataset)}")

In [ ]:
# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4)

print(f"Data loaders created")

## Initialize Preprocessor

In [ ]:
preprocessor = MultimodalPreprocessor(
    text_model_name='google-bert/bert-base-uncased',
    image_model_name='google/vit-base-patch16-224',
    clip_model_name='openai/clip-vit-base-patch32',
    max_text_length=128,
    device=device
)

print("Preprocessor initialized")

## Model A: Text-Only BERT Classifier

In [ ]:
print("="*50)
print("Training Model A: Text-Only BERT")
print("="*50)

# Create text-only model
text_model_config = ClassifierConfig(
    model_type='text_only',
    freeze_encoders=True
)
text_model = MultimodalClassifier(text_model_config)
text_model.to(device)

print(f"Model parameters: {sum(p.numel() for p in text_model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in text_model.parameters() if p.requires_grad):,}")

In [ ]:
# Training configuration
train_config = TrainingConfig(
    batch_size=16,
    epochs=5,
    learning_rate=1e-3,
    device=str(device),
    checkpoint_dir=str(CHECKPOINT_DIR / 'text_only'),
    seed=42
)

# Create trainer
trainer = Trainer(
    model=text_model,
    preprocessor=preprocessor,
    config=train_config,
    train_loader=train_loader,
    val_loader=val_loader
)

# Train
text_history = trainer.train()
print("Text-only model training complete!")

## Model B: Image-Only ViT Classifier

In [ ]:
print("="*50)
print("Training Model B: Image-Only ViT")
print("="*50)

# Create image-only model
image_model_config = ClassifierConfig(
    model_type='image_only',
    freeze_encoders=True
)
image_model = MultimodalClassifier(image_model_config)
image_model.to(device)

print(f"Model parameters: {sum(p.numel() for p in image_model.parameters()):,}")

In [ ]:
# Create trainer for image model
image_trainer = Trainer(
    model=image_model,
    preprocessor=preprocessor,
    config=train_config,
    train_loader=train_loader,
    val_loader=val_loader
)

# Train
image_history = image_trainer.train()
print("Image-only model training complete!")

## Model C: Standard Multimodal Fusion

In [ ]:
print("="*50)
print("Training Model C: Standard Multimodal Fusion")
print("="*50)

# Create standard fusion model
standard_config = ClassifierConfig(
    model_type='standard',
    freeze_encoders=True
)
standard_model = MultimodalClassifier(standard_config)
standard_model.to(device)

print(f"Model parameters: {sum(p.numel() for p in standard_model.parameters()):,}")

In [ ]:
# Create trainer for standard model
standard_trainer = Trainer(
    model=standard_model,
    preprocessor=preprocessor,
    config=train_config,
    train_loader=train_loader,
    val_loader=val_loader
)

# Train
standard_history = standard_trainer.train()
print("Standard fusion model training complete!")

## Save Baseline Results

In [ ]:
import json
from src.utils import save_json, timestamp_now

# Save training histories
results_dir = Path('results')
ensure_dir(results_dir)

baseline_results = {
    'text_only': {
        'history': text_history.__dict__,
        'best_val_f1': text_history.best_val_f1
    },
    'image_only': {
        'history': image_history.__dict__,
        'best_val_f1': image_history.best_val_f1
    },
    'standard': {
        'history': standard_history.__dict__,
        'best_val_f1': standard_history.best_val_f1
    }
}

save_json(baseline_results, results_dir / 'baseline_results.json')

print("Saved baseline results to results/baseline_results.json")

# Summary
print("\n" + "="*50)
print("BASELINE TRAINING COMPLETE")
print("="*50)
print(f"\nModel A (Text-Only) - Best Val F1: {text_history.best_val_f1:.4f}")
print(f"Model B (Image-Only) - Best Val F1: {image_history.best_val_f1:.4f}")
print(f"Model C (Standard) - Best Val F1: {standard_history.best_val_f1:.4f}")

print("\nNext: Run 04_consistency_aware_training.ipynb for the proposed model")